# Inspect per-tx Arbitrum revenue data

Two views:
1. The cached parquet at `data/onchain_blocks_transactions/per_tx.parquet` — 40M rows, 4-column schema (block-number, eff-price, l2-gas, l1-gas).
2. A live narrow ClickHouse query using the canonical `sql/arbitrum_revenue_per_tx.sql` — full schema with `tx_hash`, `block_time`, gas decomposition, and **fee decomposition (`l2_base`, `l2_surplus`, `l1_fees`, `total_fees`)**.

To regenerate the parquet at the canonical schema for the full window, run `python scripts/fetch_data.py --only per-tx --force`.

## 1. Cached parquet — block-grain confirms it's per-tx

In [ ]:
import os
import pathlib
import pandas as pd
from dotenv import dotenv_values
import clickhouse_connect

ROOT    = pathlib.Path().resolve().parent
PARQUET = ROOT / 'data' / 'onchain_blocks_transactions' / 'per_tx.parquet'
# Override via the ARBOS_ENV_PATH env var; fallback is ~/.config/arbos60/.env.
ENV     = pathlib.Path(os.environ.get(
    'ARBOS_ENV_PATH',
    str(pathlib.Path.home() / '.config' / 'arbos60' / '.env'),
))

df = pd.read_parquet(PARQUET)
print(f'rows           {len(df):>14,}')
print(f'unique blocks  {df["block_number"].nunique():>14,}')
print(f'avg txs/block  {len(df) / df["block_number"].nunique():>14.2f}')
df.dtypes

In [2]:
df.head(10)

,block_number,eff_price_gwei,l2_gas,l1_gas
0,422664553,0.020176,249180.0,0.0
1,422664553,0.020176,266304.0,0.0
2,422664554,0.020000,57207.0,0.0
3,422664553,0.020176,266304.0,0.0
4,422664553,0.020176,0.0,0.0
5,422664556,0.020128,51280.0,0.0
6,422664553,0.020176,89632.0,0.0
7,422664554,0.020000,0.0,0.0
8,422664553,0.020176,443188.0,0.0
9,422664556,0.020128,0.0,0.0


## 2. Live ClickHouse query — full schema with `tx_hash` and fee decomposition

Narrow 30-minute peak window (Jan 31 17:00–17:30 UTC) to keep this fast (~360K rows, ~2 sec).

In [3]:
cfg = dotenv_values(ENV)
client = clickhouse_connect.get_client(
    host=cfg['CLICKHOUSE_HOST'], user=cfg['CLICKHOUSE_USER'],
    password=cfg['CLICKHOUSE_PASSWORD'], port=8443, secure=True,
)
q = '''
SELECT
    block_number,
    tx_hash,
    block_time,

    -- Per-gas effective price (gwei) and gas decomposition.
    toFloat64(effective_gas_price) / 1e9                              AS eff_price_gwei,
    toFloat64(gas_used)                                               AS gas_used,
    toFloat64(gas_used_for_l1)                                        AS gas_used_for_l1,
    toFloat64(gas_used - gas_used_for_l1)                             AS l2_gas,

    -- Fee decomposition (ETH).
    -- l2_base    = P_min × g_L2  (P_min = 0.02 gwei, fixed in this range)
    -- l2_surplus = (P_eff − P_min) × g_L2
    -- l1_fees    = P_eff × g_L1
    -- total_fees = P_eff × g_total
    0.02e9 * toFloat64(gas_used - gas_used_for_l1) / 1e18             AS l2_base,
    (toFloat64(effective_gas_price) - 0.02e9)
        * toFloat64(gas_used - gas_used_for_l1) / 1e18                AS l2_surplus,
    toFloat64(effective_gas_price) * toFloat64(gas_used_for_l1) / 1e18 AS l1_fees,
    toFloat64(effective_gas_price) * toFloat64(gas_used) / 1e18       AS total_fees

FROM raw_arbitrum.v_receipts
WHERE block_time >= toDateTime('2026-01-31 17:00:00')
  AND block_time <  toDateTime('2026-01-31 17:30:00')
ORDER BY block_number, tx_hash
'''
tx = client.query_df(q)
client.close()
print(f'rows: {len(tx):,}')
print(f'cols: {list(tx.columns)}')
tx.dtypes

rows: 364,015
cols: ['block_number', 'tx_hash', 'block_time', 'eff_price_gwei', 'gas_used', 'gas_used_for_l1', 'l2_gas', 'l2_base', 'l2_surplus', 'l1_fees', 'total_fees']


block_number               uint64
tx_hash                    string
block_time         datetime64[ns]
eff_price_gwei            float64
gas_used                  float64
gas_used_for_l1           float64
l2_gas                    float64
l2_base                   float64
l2_surplus                float64
l1_fees                   float64
total_fees                float64
dtype: object

In [4]:
tx.head(10)

,block_number,tx_hash,block_time,eff_price_gwei,gas_used,gas_used_for_l1,l2_gas,l2_base,l2_surplus,l1_fees,total_fees
0,427213823,0x02d338a8f54e6113c780b329d55c74e4842e255ff267...,2026-01-31 17:00:00,0.022678,119302.0,0.0,119302.0,2.386040e-06,3.194908e-07,0.0,2.705531e-06
1,427213823,0x1d809f2f70bd7c5a8c15e07d0e947a10cc463f8d3426...,2026-01-31 17:00:00,0.022678,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00
2,427213823,0x77bc77458dab504d1ff598796655fc7574b66bba39f6...,2026-01-31 17:00:00,0.022678,45047.0,0.0,45047.0,9.009400e-07,1.206359e-07,0.0,1.021576e-06
3,427213823,0xbf06d5a8583889fcc0f7c99b5fbb94c8033e13fd4cf1...,2026-01-31 17:00:00,0.022678,39193.0,0.0,39193.0,7.838600e-07,1.049589e-07,0.0,8.888189e-07
4,427213823,0xc27e11d7fcc033c20999e957fe1674a138b044cdee38...,2026-01-31 17:00:00,0.022678,48726.0,0.0,48726.0,9.745200e-07,1.304882e-07,0.0,1.105008e-06
5,427213824,0x04cce198dca457ec164c336b21a3f6fadac9ccba98ca...,2026-01-31 17:00:00,0.022582,40283.0,0.0,40283.0,8.056600e-07,1.040107e-07,0.0,9.096707e-07
6,427213824,0x08fe186b29eef29f8bad90e2b7d9fd832d4841bab09c...,2026-01-31 17:00:00,0.022582,61129.0,0.0,61129.0,1.222580e-06,1.578351e-07,0.0,1.380415e-06
7,427213824,0x13aba181f3c0aaf89ce4557960944eb20fc47207943f...,2026-01-31 17:00:00,0.022582,45047.0,0.0,45047.0,9.009400e-07,1.163114e-07,0.0,1.017251e-06
8,427213824,0x1ee93d1ab3903e70697e9262b69b4dd3a660492cbbe6...,2026-01-31 17:00:00,0.022582,45035.0,0.0,45035.0,9.007000e-07,1.162804e-07,0.0,1.016980e-06
9,427213824,0x30e4c1f2a1c94563615ab275cd720a95313a3083b8e0...,2026-01-31 17:00:00,0.022582,45047.0,0.0,45047.0,9.009400e-07,1.163114e-07,0.0,1.017251e-06


In [5]:
# Fee-decomposition spot-checks per tx.
# total_fees should equal l2_base + l2_surplus + l1_fees within float tolerance.
tx_check = tx.assign(
    decomposed = tx['l2_base'] + tx['l2_surplus'] + tx['l1_fees'],
    diff       = (tx['l2_base'] + tx['l2_surplus'] + tx['l1_fees']) - tx['total_fees'],
)
print(f'max |total - (l2_base + l2_surplus + l1_fees)| over {len(tx_check):,} txs: {tx_check["diff"].abs().max():.2e} ETH')
tx_check[['tx_hash', 'eff_price_gwei', 'l2_gas', 'gas_used_for_l1', 'l2_base', 'l2_surplus', 'l1_fees', 'total_fees']].head(10)

max |total - (l2_base + l2_surplus + l1_fees)| over 364,015 txs: 1.73e-18 ETH


,tx_hash,eff_price_gwei,l2_gas,gas_used_for_l1,l2_base,l2_surplus,l1_fees,total_fees
0,0x02d338a8f54e6113c780b329d55c74e4842e255ff267...,0.022678,119302.0,0.0,2.386040e-06,3.194908e-07,0.0,2.705531e-06
1,0x1d809f2f70bd7c5a8c15e07d0e947a10cc463f8d3426...,0.022678,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00
2,0x77bc77458dab504d1ff598796655fc7574b66bba39f6...,0.022678,45047.0,0.0,9.009400e-07,1.206359e-07,0.0,1.021576e-06
3,0xbf06d5a8583889fcc0f7c99b5fbb94c8033e13fd4cf1...,0.022678,39193.0,0.0,7.838600e-07,1.049589e-07,0.0,8.888189e-07
4,0xc27e11d7fcc033c20999e957fe1674a138b044cdee38...,0.022678,48726.0,0.0,9.745200e-07,1.304882e-07,0.0,1.105008e-06
5,0x04cce198dca457ec164c336b21a3f6fadac9ccba98ca...,0.022582,40283.0,0.0,8.056600e-07,1.040107e-07,0.0,9.096707e-07
6,0x08fe186b29eef29f8bad90e2b7d9fd832d4841bab09c...,0.022582,61129.0,0.0,1.222580e-06,1.578351e-07,0.0,1.380415e-06
7,0x13aba181f3c0aaf89ce4557960944eb20fc47207943f...,0.022582,45047.0,0.0,9.009400e-07,1.163114e-07,0.0,1.017251e-06
8,0x1ee93d1ab3903e70697e9262b69b4dd3a660492cbbe6...,0.022582,45035.0,0.0,9.007000e-07,1.162804e-07,0.0,1.016980e-06
9,0x30e4c1f2a1c94563615ab275cd720a95313a3083b8e0...,0.022582,45047.0,0.0,9.009400e-07,1.163114e-07,0.0,1.017251e-06


In [6]:
# Per-block summary: how many txs, total surplus / total fees per block.
summary = (
    tx.groupby('block_number')
      .agg(n_txs=('tx_hash', 'size'),
           eff_price=('eff_price_gwei', 'first'),
           l2_base_eth=('l2_base', 'sum'),
           l2_surplus_eth=('l2_surplus', 'sum'),
           l1_fees_eth=('l1_fees', 'sum'),
           total_fees_eth=('total_fees', 'sum'))
      .sort_values('total_fees_eth', ascending=False)
      .head(10)
)
summary

,n_txs,eff_price,l2_base_eth,l2_surplus_eth,l1_fees_eth,total_fees_eth
block_number,,,,,,
427216887,504,1.844502,0.000796,0.072622,0.000249,0.073668
427216888,640,1.788228,0.000641,0.056636,0.000328,0.057605
427216886,661,1.765058,0.000640,0.055833,0.000322,0.056795
427216799,12,1.556562,0.000705,0.054134,0.000008,0.054847
427216885,642,1.688340,0.000639,0.053333,0.000319,0.054292
427216883,320,1.666182,0.000640,0.052649,0.000190,0.053478
427216890,496,1.972278,0.000538,0.052471,0.000256,0.053265
427216884,409,1.614206,0.000640,0.050990,0.000218,0.051848
427216801,116,1.587946,0.000646,0.050616,0.000122,0.051384


In [7]:
# Window-total fee decomposition (ETH) over the 30-minute peak.
totals = tx[['l2_base', 'l2_surplus', 'l1_fees', 'total_fees']].sum()
print(totals)
print()
print(f'l2_base    {totals["l2_base"]   :>10.4f} ETH ({totals["l2_base"]   /totals["total_fees"]:>6.1%} of total)')
print(f'l2_surplus {totals["l2_surplus"]:>10.4f} ETH ({totals["l2_surplus"]/totals["total_fees"]:>6.1%} of total)')
print(f'l1_fees    {totals["l1_fees"]   :>10.4f} ETH ({totals["l1_fees"]   /totals["total_fees"]:>6.1%} of total)')
print(f'total      {totals["total_fees"]:>10.4f} ETH')

l2_base       0.773433
l2_surplus    8.523351
l1_fees       0.151916
total_fees    9.448701
dtype: float64

l2_base        0.7734 ETH (  8.2% of total)
l2_surplus     8.5234 ETH ( 90.2% of total)
l1_fees        0.1519 ETH (  1.6% of total)
total          9.4487 ETH


## 3. Real L2 fees (on-chain) vs ArbOS 51 simulation (our methodology)

Per-block L2 fee (ETH): *real* uses on-chain effective_gas_price; *sim* runs our `arbos51.price_per_block` over the block sequence (Lindley + Taylor4 exp). Inflow = sum of per-tx L2 gas, fee = price · L2 gas — same per-tx → per-block roll-up as everywhere else.

Pulls 2 days of block-level totals from ClickHouse for the Lindley warm-up, then filters to the 30-min window for the chart.

In [8]:
import sys
sys.path.insert(0, str(ROOT / 'scripts'))
import arbos51 as a51
import numpy as np
import plotly.graph_objects as go

# Pull block-level totals (block_number, block_time, L2-only gas) from CH.
# 2 days is plenty of warm-up for the Lindley recursion before our window.
cfg = dotenv_values(ENV)
client = clickhouse_connect.get_client(
    host=cfg['CLICKHOUSE_HOST'], user=cfg['CLICKHOUSE_USER'],
    password=cfg['CLICKHOUSE_PASSWORD'], port=8443, secure=True,
)
blocks_df = client.query_df('''
    SELECT
        block_number,
        MIN(block_time)                               AS block_time,
        SUM(toFloat64(gas_used - gas_used_for_l1))    AS total_l2_gas
    FROM raw_arbitrum.v_receipts
    WHERE block_date >= toDate('2026-01-30')
      AND block_date <  toDate('2026-02-01')
    GROUP BY block_number
    ORDER BY block_number
''')
client.close()
blocks_df['block_time'] = pd.to_datetime(blocks_df['block_time']).dt.tz_localize(None)
print(f'warm-up + window: {len(blocks_df):,} blocks')

# Run ArbOS 51 sim over the full block sequence — Lindley is path-dependent
# so we need the contiguous chain to compute prices accurately at the end.
total_gas_pb = blocks_df['total_l2_gas'].to_numpy().astype(np.float64)
dt_s         = a51.dt_seconds_per_block(blocks_df['block_time'].to_numpy())
sim_p_gwei   = a51.price_per_block(total_gas_pb, dt_s)
block_to_sim = dict(zip(blocks_df['block_number'].to_numpy(), sim_p_gwei))
print(f'sim ArbOS 51 prices computed for all {len(sim_p_gwei):,} blocks')

warm-up + window: 694,345 blocks
sim ArbOS 51 prices computed for all 694,345 blocks


In [9]:
# Per-tx → per-block roll-up of L2 fees in our 30-min sample,
# computed with the full methodology (per-tx fee = price × l2_gas).
tx2 = tx.copy()
tx2['sim_p51_gwei']    = tx2['block_number'].map(block_to_sim)
tx2['real_l2_fee_eth'] = tx2['l2_base'] + tx2['l2_surplus']                     # P_eff · L2_gas / 1e18
tx2['sim_l2_fee_eth']  = tx2['sim_p51_gwei'] * 1e9 * tx2['l2_gas'] / 1e18         # P_sim · L2_gas

per_block = (
    tx2.groupby('block_number', as_index=False)
       .agg(block_time      =('block_time',      'first'),
            real_l2_fee_eth =('real_l2_fee_eth', 'sum'),
            sim_l2_fee_eth  =('sim_l2_fee_eth',  'sum'),
            real_p_gwei     =('eff_price_gwei',  'first'),
            sim_p_gwei      =('sim_p51_gwei',    'first'))
       .sort_values('block_time')
)

real_total = per_block['real_l2_fee_eth'].sum()
sim_total  = per_block['sim_l2_fee_eth'].sum()
print(f'30-min window L2 fee total (ETH):')
print(f'  real on-chain  {real_total:>10.4f}')
print(f'  arbos51 sim    {sim_total:>10.4f}')
print(f'  diff           {sim_total - real_total:>+10.4f}  ({(sim_total - real_total)/real_total:+.3%})')

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=per_block['block_time'], y=per_block['real_l2_fee_eth'],
    mode='lines', name='Real L2 fee (on-chain)',
    line=dict(color='#d62728', width=1.6),
    hovertemplate='%{x|%H:%M:%S} · real=%{y:.4f} ETH<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=per_block['block_time'], y=per_block['sim_l2_fee_eth'],
    mode='lines', name='ArbOS 51 sim L2 fee',
    line=dict(color='#1f77b4', width=1.6, dash='dash'),
    hovertemplate='%{x|%H:%M:%S} · sim=%{y:.4f} ETH<extra></extra>',
))
fig.update_layout(
    title=(f'Per-block L2 fees: real on-chain vs ArbOS 51 sim<br>'
           f'<sub>Jan 31 17:00–17:30 UTC · {len(per_block):,} blocks · {len(tx2):,} txs · '
           f'sim − real = {(sim_total - real_total)/real_total:+.3%}</sub>'),
    xaxis_title='block time', yaxis_title='L2 fees (ETH)',
    template='plotly_white', height=420, hovermode='x unified',
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig.show()

30-min window L2 fee total (ETH):
  real on-chain      9.2968
  arbos51 sim        9.4869
  diff              +0.1901  (+2.045%)
